In [1]:
import pandas as pd
import numpy as np
import os

EXPORT_PATH = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\exports"

master = pd.read_csv(os.path.join(EXPORT_PATH, "master_orders.csv"),
                     parse_dates=['order_purchase_timestamp',
                                  'order_delivered_customer_date',
                                  'order_estimated_delivery_date'])

print(f"✅ Master loaded: {master.shape}")

✅ Master loaded: (96470, 23)


In [2]:
# Reference date = 1 day after last order in dataset
reference_date = master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = master.groupby('customer_unique_id').agg(
    last_purchase = ('order_purchase_timestamp', 'max'),
    frequency     = ('order_id', 'nunique'),
    monetary      = ('total_revenue', 'sum')
).reset_index()

rfm['recency'] = (reference_date - rfm['last_purchase']).dt.days
rfm = rfm.drop(columns='last_purchase')

print(rfm.head())
print(f"\nShape: {rfm.shape}")

                 customer_unique_id  frequency  monetary  recency
0  0000366f3b9a7992bf8c76cfdf3221e2          1    129.90      112
1  0000b849f77a49e4a4ce2b2a4ca5be3f          1     18.90      115
2  0000f46a3911fa3c0805444483337064          1     69.00      537
3  0000f6ccb0745a6a4b88665a16c9f078          1     25.99      321
4  0004aac84e0df4da2b147fca70cf8255          1    180.00      288

Shape: (93350, 4)


In [3]:
# Higher recency score = bought MORE recently
rfm['R_score'] = pd.qcut(rfm['recency'],  q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

print(rfm.head(10))

                 customer_unique_id  frequency  monetary  recency  R_score  \
0  0000366f3b9a7992bf8c76cfdf3221e2          1    129.90      112        4   
1  0000b849f77a49e4a4ce2b2a4ca5be3f          1     18.90      115        4   
2  0000f46a3911fa3c0805444483337064          1     69.00      537        1   
3  0000f6ccb0745a6a4b88665a16c9f078          1     25.99      321        2   
4  0004aac84e0df4da2b147fca70cf8255          1    180.00      288        2   
5  0004bd2a26a76fe21f786e4fbd80607f          1    154.00      146        4   
6  00050ab1314c0e55a6ca13cf7181fecf          1     27.99      132        4   
7  00053a61a98854899e70ed204dd4bafe          1    382.00      183        3   
8  0005e1862207bf6ccc02e4228effd9a0          1    135.00      543        1   
9  0005ef4cd20d2893f0d9fbd94d3c0d97          1    104.90      170        4   

   F_score  M_score RFM_score  
0        1        4       414  
1        1        1       411  
2        1        2       112  
3        1   

In [4]:
def segment_customer(row):
    r = row['R_score']
    f = row['F_score']
    m = row['M_score']

    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Cannot Lose Them'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

print("\n📊 Customer Segment Counts:")
print(rfm['segment'].value_counts())

rfm.to_csv(os.path.join(EXPORT_PATH, "rfm_segments.csv"), index=False)
print("\n✅ rfm_segments.csv saved!")


📊 Customer Segment Counts:
segment
At Risk                22228
Loyal Customers        18823
New Customers          14980
Champions              14959
Cannot Lose Them        8723
Potential Loyalists     7374
Lost                    6263
Name: count, dtype: int64

✅ rfm_segments.csv saved!


In [5]:
# Get each customer's first purchase month
cohort_data = master[['customer_unique_id', 'order_purchase_timestamp']].copy()

cohort_data['order_month'] = cohort_data['order_purchase_timestamp'].dt.to_period('M')

first_purchase = cohort_data.groupby('customer_unique_id')['order_month'].min().reset_index()
first_purchase.columns = ['customer_unique_id', 'cohort_month']

cohort_data = cohort_data.merge(first_purchase, on='customer_unique_id')

# Calculate period number (0 = first month, 1 = second month, etc.)
cohort_data['period'] = (
    cohort_data['order_month'] - cohort_data['cohort_month']
).apply(lambda x: x.n)

print(cohort_data.head())

                 customer_unique_id order_purchase_timestamp order_month  \
0  7c396fd4830fd04220f754e42b4e5bff      2017-10-02 10:56:33     2017-10   
1  af07308b275d755c9edb36a90c618231      2018-07-24 20:41:37     2018-07   
2  3a653a41f6f9fc3d2a113cf8398680e8      2018-08-08 08:38:49     2018-08   
3  7c142cf63193a1473d2e66489a9ae977      2017-11-18 19:28:06     2017-11   
4  72632f0f9dd73dfee390c9b22eb56dd6      2018-02-13 21:18:39     2018-02   

  cohort_month  period  
0      2017-09       1  
1      2018-07       0  
2      2018-08       0  
3      2017-11       0  
4      2018-02       0  


In [6]:
cohort_pivot = cohort_data.groupby(['cohort_month', 'period'])['customer_unique_id'].nunique().reset_index()
cohort_pivot.columns = ['cohort_month', 'period', 'customers']

cohort_matrix = cohort_pivot.pivot_table(index='cohort_month', columns='period', values='customers')

# Retention rate (% of original cohort)
cohort_size    = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_size, axis=0).round(3) * 100

print("📊 Retention Matrix (first 5 cohorts, first 6 periods):")
print(retention_matrix.iloc[:5, :6])

cohort_matrix.to_csv(os.path.join(EXPORT_PATH, "cohort_matrix.csv"))
retention_matrix.to_csv(os.path.join(EXPORT_PATH, "retention_matrix.csv"))
print("\n✅ Cohort files saved!")

📊 Retention Matrix (first 5 cohorts, first 6 periods):
period            0      1    2    3    4    5
cohort_month                                  
2016-09       100.0    NaN  NaN  NaN  NaN  NaN
2016-10       100.0    NaN  NaN  NaN  NaN  NaN
2016-12       100.0  100.0  NaN  NaN  NaN  NaN
2017-01       100.0    0.3  0.3  0.1  0.4  0.1
2017-02       100.0    0.2  0.3  0.1  0.4  0.1

✅ Cohort files saved!


In [7]:
monthly = master.groupby('order_year_month').agg(
    revenue  = ('total_revenue', 'sum'),
    orders   = ('order_id', 'nunique'),
    customers= ('customer_unique_id', 'nunique')
).reset_index().sort_values('order_year_month')

# Remove first and last month (usually incomplete data)
monthly = monthly.iloc[1:-1]

# Month-over-month growth
monthly['revenue_mom_pct'] = monthly['revenue'].pct_change() * 100

print("📅 Monthly Summary:")
print(monthly.to_string(index=False))

monthly.to_csv(os.path.join(EXPORT_PATH, "monthly_trends.csv"), index=False)
print("\n✅ monthly_trends.csv saved!")

📅 Monthly Summary:
order_year_month   revenue  orders  customers  revenue_mom_pct
         2016-10  40325.11     265        262              NaN
         2016-12     10.90       1          1    -9.997297e+01
         2017-01 111798.36     750        718     1.025573e+06
         2017-02 234223.40    1653       1630     1.095052e+02
         2017-03 359198.85    2546       2508     5.335737e+01
         2017-04 340669.68    2303       2274    -5.158471e+00
         2017-05 489159.25    3545       3478     4.358755e+01
         2017-06 421923.37    3135       3076    -1.374519e+01
         2017-07 481604.52    3872       3802     1.414502e+01
         2017-08 554699.70    4193       4114     1.517743e+01
         2017-09 607399.67    4150       4083     9.500631e+00
         2017-10 648247.65    4478       4417     6.725058e+00
         2017-11 987648.07    7288       7182     5.235660e+01
         2017-12 726033.19    5513       5450    -2.648867e+01
         2018-01 924645.00    7069  

In [8]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow = master.groupby('order_dow').agg(
    orders  = ('order_id', 'nunique'),
    revenue = ('total_revenue', 'sum')
).reindex(dow_order).reset_index()

print("📅 Orders by Day of Week:")
print(dow.to_string(index=False))

dow.to_csv(os.path.join(EXPORT_PATH, "dow_trends.csv"), index=False)
print("\n✅ dow_trends.csv saved!")

📅 Orders by Day of Week:
order_dow  orders    revenue
   Monday   15701 2168905.61
  Tuesday   15502 2122147.22
Wednesday   15074 2051158.81
 Thursday   14322 1958421.49
   Friday   13684 1910385.13
 Saturday   10555 1464049.60
   Sunday   11632 1545181.07

✅ dow_trends.csv saved!
